# GLARE Methodology: pipeline runner (Assignment 2, Group 1)

**Status: skeleton.** This notebook only runs the modules in the repository, one stage per section, and shows the result of each stage. The logic lives in `glare_issues/`. Each section is owned by the member named in its heading, who edits their own cell through a pull request.

**Rules**
- Every number reported in the final document must come from a cell of this notebook, never from the demo mode.
- Same seed (42) everywhere. LLM calls at temperature 0, answers cached on disk.
- Save the output of every stage to Google Drive (last cell of each section), so a disconnected Colab can continue from the last checkpoint.
- Use a GPU for the embedding stage: Runtime, Change runtime type, T4 GPU.

**Open decisions (fill in, then delete this line):** LLM name and version = `TODO`, Model C checkpoint = `TODO`, sample size = `TODO`.

## 0. Setup (everyone)

In [ ]:
# Settings: change only after agreeing in the group chat
DEMO = True               # True = synthetic data, mock LLM, offline embedder (smoke test, NOT results). Set False for real runs.
REPO_URL = 'https://github.com/layan-sa1/glare-issue-discovery.git'   # TODO: check the exact link
DRIVE_DIR = '/content/drive/MyDrive/GLARE_Project'                     # where checkpoints are saved
MASTER_POOL_ZIP = DRIVE_DIR + '/GLARE_Master_Pool_1M.zip'               # same file used in the pilot
TARGET_SAMPLE = 30_000    # TODO: agree on the final sample size
LLM_BACKEND = 'anthropic' # 'anthropic' or 'openai' (ignored when DEMO = True)
LLM_MODEL = ''            # TODO: exact model name and version, also write it in the report
SEED = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# Get the code. A private repository needs a GitHub token saved in Colab Secrets (key icon on the left) under the name GH_TOKEN.
from google.colab import userdata
token = userdata.get('GH_TOKEN')
url = REPO_URL.replace('https://', f'https://{token}@')
!git clone -q {url} repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# If you changed code on GitHub after cloning, run this to update:
# !git pull

In [ ]:
import sys, importlib
sys.path.insert(0, '.')
import pandas as pd, numpy as np
import run_pipeline as R
from glare_issues.config import Config
from glare_issues.llm import make_llm

cfg = Config(out_dir='results', llm_backend=LLM_BACKEND, llm_model=LLM_MODEL, target_sample=TARGET_SAMPLE)
cfg.seed = SEED
embedders = cfg.embedders
if DEMO:
    from glare_issues.demo_data import make_demo_reviews
    cfg.out_dir, cfg.cache_dir, cfg.llm_backend = 'demo_results', 'demo_cache', 'mock'
    cfg.n_trials, cfg.kmeans_k_range = 8, [5, 7, 10, 15]
    cfg.umap_space = {'n_neighbors': [10, 15], 'n_components': [5, 10], 'min_dist': [0.0, 0.1]}
    cfg.hdbscan_space = {'min_cluster_size': [10, 20, 40], 'min_samples': [1, 5], 'cluster_selection_method': ['eom', 'leaf']}
    embedders = {'DemoTFIDF-A': {'kind': 'tfidf', 'name': ''}}
    print('DEMO MODE: these numbers are NOT research results.')
llm = make_llm(cfg)
print('Output folder:', cfg.out_dir)

## 1. Data and preprocessing (Member 1)
Reads the Master Pool, cleans, merges duplicates (keeping counts), applies the per-app cap. Output: `01_preprocessed.csv`.

TODO (Member 1): read `MASTER_POOL_ZIP` in chunks (as in the pilot notebook) into a dataframe `df` with columns `review_id, app_id, content, rating, at`. Report how many apps entered the sample and how many reviews each contributed.

In [ ]:
if DEMO:
    df = make_demo_reviews()
else:
    # TODO (Member 1): load the real sample here, for example from the Master Pool ZIP in chunks
    raise NotImplementedError('Load the real GLARE sample into df')
d = R.stage_prepare(cfg, df)
display(d.head())

In [ ]:
# Checkpoint to Drive
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 2. Stage 1: problem filter (Member 5)
LLM classifies each unique review as problem report or not. Outputs: `02_filtered_all.csv`, `02_problem_reviews.csv`.

TODO (Member 5): choose the LLM, run a 200-review pilot against human labels, record the model name and version.

In [ ]:
d_all, kept = R.stage_filter(cfg, d, llm)
display(kept.head())

In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 3. Annotation sheets (Member 6)
Writes blank sheets for the human parts of the evaluation (filter labels, issue grouping). Annotators fill the last column and save the file with the name plus `_done`, in `results/annotations/`.

In [ ]:
R.stage_annotate(cfg, d_all, kept)

In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 4. Embeddings, UMAP, HDBSCAN, K-Means (Members 2, 3, 4)
Members 2 and 3 own the embedding models A, B and C (`glare_issues/embed.py`, settings in `config.py`). Member 4 owns UMAP, HDBSCAN, K-Means and the random search (`glare_issues/cluster.py`). Needs a GPU for real runs.

TODO (Member 3): replace the placeholder Model C checkpoint in `config.py`.

In [ ]:
table, store = R.stage_cluster(cfg, kept, embedders)
display(table)

In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 5. Stage 3: issue descriptions and ranking (Member 5)
Representative reviews, LLM label and Arabic summary, ranking by occurrences. Output: `04_ranked_issues.csv`.

In [ ]:
res = R.stage_summarize(cfg, kept, store, table, llm)
display(res.head(10))

In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 6. Evaluation (Member 6)
Reads the completed annotation sheets (`*_done.csv`). If a sheet is missing, that part is skipped. Output: `05_evaluation_report.json`, `05_external_scores.csv`, figures.

TODO (Member 6): copy the numbers into the result tables of the report (Tables in Section 6.5) and into the benchmarking comparison.

In [ ]:
report = R.stage_evaluate(cfg, kept, d_all)
report

In [ ]:
!cp -r {cfg.out_dir} {DRIVE_DIR}/

## 7. Final check (everyone)
- [ ] `DEMO = False`, and the whole notebook runs from the first cell to the last on a fresh Colab session.
- [ ] LLM name and version, Model C checkpoint and sample size are filled in at the top and written in the report.
- [ ] Every table and figure in the report comes from this notebook.
- [ ] No API keys or tokens appear in any cell output.